In [124]:
import pandas as pd
import tkinter as tk
import numpy as np


In [125]:
df = pd.read_csv("poke_data.csv",delimiter="/", header=None, names=["name", "is_legendary", "color", "generation", "habitat", "evo_stage", "types", "base_experience", "height"])

In [126]:
df

,name,is_legendary,color,generation,habitat,evo_stage,types,base_experience,height
0,bulbasaur,False,green,generation-i,grassland,base,"['grass', 'poison']",64,7
1,ivysaur,False,green,generation-i,grassland,middle,"['grass', 'poison']",142,10
2,venusaur,False,green,generation-i,grassland,final,"['grass', 'poison']",236,20
3,charmander,False,red,generation-i,mountain,base,['fire'],62,6
4,charmeleon,False,red,generation-i,mountain,middle,['fire'],142,11
...,...,...,...,...,...,...,...,...,...
986,raging-bolt,False,yellow,generation-ix,NaN,no-evolution,"['electric', 'dragon']",295,52
987,iron-boulder,False,gray,generation-ix,NaN,no-evolution,"['rock', 'psychic']",295,15
988,iron-crown,False,blue,generation-ix,NaN,no-evolution,"['steel', 'psychic']",295,16
989,terapagos,True,blue,generation-ix,NaN,no-evolution,['normal'],90,2


In [127]:
class pokemon:
    
    def __init__(self,row):
        self.name = row["name"]
        self.is_legendary = row["is_legendary"]
        self.color = row["color"]
        self.generation = row["generation"]
        self.habitat = row["habitat"]
        self.evo_stage = row["evo_stage"]
        self.types = row["types"]
        self.base_experience = row["base_experience"]
        self.height = row["height"]
        
    def __eq__(self, other):
        if self.name == other.name:
            return True
        else:
            return False
        
    def compare_others_attributes(self,other):
        
        both_legendary = self.is_legendary == other.is_legendary
        both_color = self.color == other.color
        both_generation = self.generation == other.generation
        both_habitat = self.habitat == other.habitat
        both_evo_stage = self.evo_stage == other.evo_stage
        both_types = self.types == other.types
        both_base_experience = self.base_experience == other.base_experience
        both_height = self.height == other.height
        others_attribute_info = [both_legendary,both_color,both_generation,both_habitat,both_evo_stage,both_types,both_base_experience,both_height]
        return others_attribute_info ### compares the attributes of one pokemon to another i.e do they share the same legendary status etc
    

In [142]:
#conditions assisted by Claude
def process_attribute_comparison(attributes, pokemon):
    """
    Converts boolean attribute matches into readable comparison hints.
    attributes: list of booleans from compare_others_attributes()
    pokemon: the selected pokemon (not the target)
    """

    # --- Legendary ---
    if attributes[0] and pokemon.is_legendary:
        legendary_comparison = "Both Pokémon are legendary"
    elif attributes[0] and not pokemon.is_legendary:
        legendary_comparison = "Neither Pokémon is legendary"
    elif not attributes[0] and pokemon.is_legendary:
        legendary_comparison = "Target Pokémon is not legendary"
    else:
        legendary_comparison = "Target Pokémon is legendary"

    # --- Color ---
    if attributes[1]:
        color_comparison = f"Both Pokémon share the same color ({pokemon.color})"
    else:
        color_comparison = f"Target Pokémon is not {pokemon.color}"

    # --- Generation ---
    if attributes[2]:
        gen_comparison = f"Both Pokémon are from Generation {str(pokemon.generation).replace('generation-', '').title()}"
    else:
        gen_comparison = f"Target Pokémon is not from Generation {str(pokemon.generation).replace('generation-', '').title()}"

    # --- Habitat ---
    if attributes[3]:
        if not pd.isna(pokemon.habitat):
            habitat_comparison = f"Both Pokémon live in the {pokemon.habitat} habitat"
        else:
            habitat_comparison = "Both Pokémon do not have a habitat specified"
    else:
        if not pd.isna(pokemon.habitat):
            habitat_comparison = f"Target Pokémon does not live in the {pokemon.habitat} habitat"
        else:
            habitat_comparison = "Target Pokémon does not have the same habitat"

    # --- Evolution stage ---
    if attributes[4]:
        evo_comparison = f"Both Pokémon are at the same evolution stage ({pokemon.evo_stage})"
    else:
        evo_comparison = f"Target Pokémon is not at evolution stage {pokemon.evo_stage}"

    # --- Types ---
    # Handle types that may be stored as list representation "['type1', 'type2']" or as string "type1/type2"
    types_raw = str(pokemon.types)
    # Remove brackets and quotes if present
    types_clean = types_raw.replace('[', '').replace(']', '').replace("'", '').replace('"', '')
    # Split by comma or slash
    types_list = [t.strip().title() for t in types_clean.split(',') if t.strip()] if ',' in types_clean else [t.strip().title() for t in types_clean.split('/') if t.strip()]
    types_str = '/'.join(types_list) if types_list else "Unknown"
    
    if attributes[5]:
        types_comparison = f"Both Pokémon share the same typing {types_str.lower()}"
    else:
        types_comparison = f"Target Pokémon does not have typing {types_str.lower()}"

    # --- Base experience ---
    if attributes[6]:
        exp_comparison = f"Both Pokémon have the same base experience ({pokemon.base_experience})"
    else:
        exp_comparison = f"Target Pokémon has different base experience than {pokemon.base_experience}"

    # --- Height ---
    if attributes[7]:
        height_comparison = f"Both Pokémon have the same height ({pokemon.height})"
    else:
        height_comparison = f"Target Pokémon has different height than {pokemon.height}"

    return [
        legendary_comparison,
        color_comparison,
        gen_comparison,
        habitat_comparison,
        evo_comparison,
        types_comparison,
        exp_comparison,
        height_comparison,
    ]

In [143]:
###CLI VERSION
target_row = df.sample().iloc[0]
target_pokemon = pokemon(target_row)
tries = 5
while(tries > 0):
    name = input("Enter a pokemon name").lower().strip()
    if(name in ["quit","q","exit","leave","stop"]):
        break
    row = df[df["name"] == name]
    if(row.empty):
        print("Selected pokemon is not in base pokedex!")
        tries -=1
    else:
        selected_pokemon = pokemon(row.iloc[0])
        print(selected_pokemon.habitat)
        if target_pokemon.__eq__(selected_pokemon) == True:
            print("congrats you win")
            break
        elif target_pokemon.__eq__(selected_pokemon) == False:
            print("incorrect guess")
            attributes = target_pokemon.compare_others_attributes(selected_pokemon)
            print(process_attribute_comparison(attributes,selected_pokemon))
            tries -=1

print(f"Sorry you lose, the correct pokemon was ...{target_pokemon.name}")
        
        

forest
incorrect guess
['Neither Pokémon is legendary', 'Target Pokémon is not yellow', 'Target Pokémon is not from Generation I', 'Both Pokémon live in the forest habitat', 'Target Pokémon is not at evolution stage middle', 'Target Pokémon does not have typing electric', 'Target Pokémon has different base experience than 112', 'Target Pokémon has different height than 4']


KeyboardInterrupt: Interrupted by user

In [111]:
def assisted_lookup(substring):
    matches = df[df['name'].str.contains(substring)]
    return matches["name"]

In [145]:
## CLAUDE GENERATED GUI VERSION
# Random target
from time import time


target_row = df.sample().iloc[0]
target_pokemon = pokemon(target_row)

tries = 5
guess_history = []  # Track all guesses

root = tk.Tk()
root.title("Pokémondle")
root.geometry("700x700")
root.resizable(False, False)

# Color scheme
BG_COLOR = "#f0f0f0"
PRIMARY_COLOR = "#FF6B35"
TEXT_COLOR = "#333333"
ACCENT_COLOR = "#004E89"
ERROR_COLOR = "#D32F2F"
SUCCESS_COLOR = "#4CAF50"

root.config(bg=BG_COLOR)

# Configure grid weights for responsiveness
root.columnconfigure(0, weight=1)

# Title
title = tk.Label(root, text="Pokémondle", font=("Arial", 32, "bold"), 
                 bg=BG_COLOR, fg=PRIMARY_COLOR)
title.grid(row=0, column=0, padx=20, pady=15)

# Tries label - styled
tries_label = tk.Label(root, text=f"Attempts Left: {tries}", 
                      font=("Arial", 13, "bold"), bg=BG_COLOR, fg=ACCENT_COLOR)
tries_label.grid(row=1, column=0, padx=20, pady=(0, 10))

# Input frame for better organization
input_frame = tk.Frame(root, bg=BG_COLOR)
input_frame.grid(row=2, column=0, padx=20, pady=10, sticky="ew")
input_frame.columnconfigure(0, weight=1)

# Entry label
entry_label = tk.Label(input_frame, text="Guess a Pokémon:", 
                      font=("Arial", 11), bg=BG_COLOR, fg=TEXT_COLOR)
entry_label.pack(anchor="w", pady=(0, 5))

# Input wrapper
input_wrapper = tk.Frame(input_frame, bg=BG_COLOR)
input_wrapper.pack(fill="x")
input_wrapper.columnconfigure(0, weight=1)

# Entry box - styled
entry1 = tk.Entry(input_wrapper, font=("Arial", 12), width=30)
entry1.grid(row=0, column=0, padx=(0, 10), sticky="ew")

# Suggestions frame with listbox
suggestions_frame = tk.Frame(input_frame, bg="white", relief="solid", bd=1)
suggestions_frame.pack(fill="both", expand=False, pady=(5, 0), ipady=5)

# Scrollbar for suggestions
suggestions_scrollbar = tk.Scrollbar(suggestions_frame, orient="vertical")
suggestions_scrollbar.pack(side="right", fill="y")

# Listbox for suggestions
suggestions_listbox = tk.Listbox(suggestions_frame, font=("Arial", 10), 
                                bg="white", fg=TEXT_COLOR, 
                                yscrollcommand=suggestions_scrollbar.set,
                                activestyle="dotbox", height=5, width=50)
suggestions_listbox.pack(side="left", fill="both", expand=True)
suggestions_scrollbar.config(command=suggestions_listbox.yview)

def update_suggestions(event=None):
    """Update suggestions based on current entry text."""
    try:
        suggestions_listbox.delete(0, tk.END)
        
        current_text = entry1.get().lower().strip()
        
        if len(current_text) == 0:
            return
        
        # Get matching pokémon using assisted_lookup
        matches = assisted_lookup(current_text)
        
        # Add matches to listbox (limit to 5 for cleaner display)
        for pokemon_name in matches.values[:5]:
            suggestions_listbox.insert(tk.END, pokemon_name)
    except Exception as e:
        print(f"Error updating suggestions: {e}")

def select_suggestion(event=None):
    """Select a suggestion and fill the entry box."""
    try:
        selection = suggestions_listbox.curselection()
        if selection:
            selected_text = suggestions_listbox.get(selection[0])
            entry1.delete(0, tk.END)
            entry1.insert(0, selected_text)
            suggestions_listbox.delete(0, tk.END)
    except Exception as e:
        print(f"Error selecting suggestion: {e}")

# Bind entry updates to suggestions on key release
entry1.bind("<KeyRelease>", lambda e: update_suggestions())
# Bind double-click on suggestion to select it
suggestions_listbox.bind("<Double-Button-1>", select_suggestion)
# Also allow Enter key to select highlighted suggestion
suggestions_listbox.bind("<Return>", select_suggestion)

entry1.focus()

# History label
history_label = tk.Label(root, text="Guess History:", 
                        font=("Arial", 11, "bold"), bg=BG_COLOR, fg=TEXT_COLOR)
history_label.grid(row=3, column=0, padx=20, pady=(15, 5), sticky="w")

# History text widget with scrollbar
history_frame = tk.Frame(root, bg="white", relief="solid", bd=1)
history_frame.grid(row=4, column=0, padx=20, pady=(0, 15), sticky="nsew")
history_frame.columnconfigure(0, weight=1)
history_frame.rowconfigure(0, weight=1)

# Scrollbar
scrollbar = tk.Scrollbar(history_frame, orient="vertical")
scrollbar.grid(row=0, column=1, sticky="ns")

# Text widget for history
history_text = tk.Text(history_frame, font=("Arial", 9), 
                      height=15, width=80, state="disabled",
                      yscrollcommand=scrollbar.set, bg="white", fg=TEXT_COLOR)
history_text.grid(row=0, column=0, sticky="nsew")
scrollbar.config(command=history_text.yview)

# Configure row 4 to expand
root.rowconfigure(4, weight=1)

def add_to_history(text, tag="normal"):
    """Add text to the history display."""
    history_text.config(state="normal")
    history_text.insert("end", text + "\n", tag)
    history_text.see("end")  # Auto-scroll to bottom
    history_text.config(state="disabled")

# Configure text tags for styling
history_text.tag_config("correct", foreground=SUCCESS_COLOR, font=("Arial", 10, "bold"))
history_text.tag_config("incorrect", foreground=ERROR_COLOR, font=("Arial", 10, "bold"))
history_text.tag_config("invalid", foreground=ERROR_COLOR)
history_text.tag_config("hint", foreground=ACCENT_COLOR, font=("Arial", 9))
history_text.tag_config("divider", foreground="#999999")

def submit_guess():
    global tries

    guess = entry1.get().lower().strip()
    entry1.delete(0, tk.END)
    suggestions_listbox.delete(0, tk.END)  # Clear suggestions after submission

    if not guess:
        add_to_history("⚠️  Please enter a Pokémon name!", "invalid")
        return

    row = df[df["name"] == guess]

    if row.empty:
        add_to_history(f"❌ {guess.capitalize()} - Not in Pokédex!", "invalid")
        return

    selected = pokemon(row.iloc[0])

    # Win condition
    if selected == target_pokemon:
        add_to_history(f"✅ {guess.capitalize()} - Correct! You win!", "correct")
        submit_btn.config(state="disabled")
        entry1.config(state="disabled")
        return

    # Incorrect guess
    tries -= 1
    tries_label.config(text=f"Attempts Left: {tries}")

    # Add guess to history
    add_to_history(f"❌ {guess.capitalize()}", "incorrect")

    attributes = target_pokemon.compare_others_attributes(selected)
    hint = process_attribute_comparison(attributes, selected)
    
    # Add hints with indentation
    for hint_text in hint:
        add_to_history(f"   • {hint_text}", "hint")
    
    add_to_history("", "divider")  # Add spacing

    if tries == 0:
        add_to_history(f"💀 Game Over! The Pokémon was: {target_pokemon.name}", "incorrect")
        submit_btn.config(state="disabled")
        entry1.config(state="disabled")
        time.sleep(2)
        root.destroy()  # Close the window after showing game over message

# Now create the submit button (after function is defined)
submit_btn = tk.Button(input_wrapper, text="Submit", font=("Arial", 11, "bold"),
                      bg=PRIMARY_COLOR, fg="white", command=submit_guess,
                      padx=20, pady=5, relief="raised", bd=1, 
                      cursor="hand2", activebackground="#E55100")
submit_btn.grid(row=0, column=1)

# Enter key support
root.bind("<Return>", lambda event: submit_guess())

root.mainloop()